In [ ]:
# step 1: Install necessary libraries
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together gradio 6.0.2

%pip install langchain-chroma==1.0.0
%pip install -q chromadb==1.3.5
%pip install -q sentence-transformers==5.1.2

In [ ]:
# Step 2: Import necessary libraries
import re
import uuid
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

COLLECTION_NAME = "semantic_cache"
client = chromadb.Client()

# Clean start when running all cells
try:
    client.delete_collection(COLLECTION_NAME)
    print("✓ Cleared existing semantic_cache collection")
except Exception:
    print("✓ No existing collection to clear")

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

def _new_id():
    return str(uuid.uuid4())

In [ ]:
 #Step 3: Cache Entry Structure
class SemanticCache:
    def __init__(self, embedder_model='all-MiniLM-L6-v2', collection_ref=None):
        """Initialize embedding model and reuse the shared ChromaDB collection."""
        self.embedder = SentenceTransformer(embedder_model)
        self.collection = collection_ref if collection_ref is not None else collection

    def add(self, query, soln_path):
        """Add query-solution path pair to cache"""
        embedding = self.embedder.encode(query).tolist()
        self.collection.add(
            embeddings=[embedding],
            documents=[query],
            metadatas=[{'query': query, 'soln_path': soln_path}],
            ids=[_new_id()]
        )

    def search(self, query, threshold=0.75):
        """Search for similar cached query"""
        embedding = self.embedder.encode(query).tolist()
        results = self.collection.query(query_embeddings=[embedding], n_results=1)
        
        if results.get('distances') and results['distances'][0]:
            score = 1 - results['distances'][0][0]
            if score >= threshold:
                metadata = results['metadatas'][0][0]
                return {
                    'soln_path': metadata['soln_path'],
                    'score': score,
                    'cached_query': metadata.get('query')
                }
        return None

# Test it
cache = SemanticCache()
cache.add("What is the capital of France?", "lookup_fact('France', 'capital')")
cache.add("How do I reset my password?", "get_help_article('password_reset')")
cache.add("What are your business hours?", "get_business_info('hours')")

test_queries = [
    "What's the capital of France?",
    "Password reset instructions",
    "When are you open?",
    "What's the weather today?"
]

for q in test_queries:
    # Get the raw similarity score even if below threshold
    embedding = cache.embedder.encode(q).tolist()
    results = cache.collection.query(query_embeddings=[embedding], n_results=1)
    
    if results.get('distances') and results['distances'][0]:
        score = 1 - results['distances'][0][0]
        
        # Now check against threshold
        result = cache.search(q)
        if result:
            print(f"✅ '{q}' → '{result['soln_path']}' (score: {result['score']:.2f})")
        else:
            print(f"❌ '{q}' → Match below threshold (score: {score:.2f})")
    else:
        print(f"❌ '{q}' → No match (no cached queries)")